# 02C - One Training Skeleton, Three Language Objectives

**Guiding question:** How much of a PyTorch language-training program can remain unchanged when the prediction job changes from context-to-center, to center-to-context, to sequence-to-sequence?

> **What you finished last time:** 02B showed that IDs address trainable rows and that supplied prediction pairs can move those rows.
>
> **The blocker:** those pairs were handed to the model. A language learner needs to manufacture lessons from ordinary text, and a useful training skeleton should not need to be rewritten for every new objective.
>
> **Endpoint:** one documented `(input IDs, target IDs)` batch contract and one reusable `fit` loop used by three deliberately small models: CBOW, Skip-gram, and a teacher-forced encoder-decoder.

## Roadmap

| Complaint | Smallest response | Evidence |
|---|---|---|
| Text is not a training batch | Tokenize, count, and assign stable IDs | Printed vocabulary and encoded documents |
| Documents are not input-target pairs | Slide a fixed-width window | Visible context/target rows and shapes |
| IDs carry no numeric features | Look up trainable rows | Tensor shape trace |
| Every model seems to demand a new loop | Give models one `training_loss` contract | CBOW uses the shared `fit` function |
| CBOW predicts one center from many neighbours | Reverse each lesson into center-to-neighbour pairs | Skip-gram trains with the same loader and loop |
| One target word is not a generated sequence | Let an encoder hand state to a decoder | Sequence logits and greedy continuation |
| Static objectives still discard token roles | Permute one CBOW context | Identical pooled representation and logits |

**Scope boundary:** CBOW and Skip-gram use full-vocabulary cross-entropy rather than negative sampling. The encoder-decoder uses fixed-length, already aligned examples, full teacher forcing during training, and no attention or padding. These toy runs prove code and shape contracts; they do not establish useful production semantics, translation quality, or free-running language ability.

## 1 - Keep the corpus small enough to inspect

The original script downloads thousands of news rows. That is useful for richer neighbours, but it hides the mechanics behind network access and data volume. Here you keep the legal phrases from 02B so every token can be traced.

The setup cell fixes every random source used in this notebook. Determinism does not make training universally reproducible across every device and PyTorch build, but it makes this CPU teaching run stable enough to test.

The tokenizer deliberately keeps only lowercase alphabetic words and hyphens. That simple rule is not a recommendation for production tokenization; it removes punctuation questions so you can focus on how context windows become gradients.

In [ ]:
import math
import re
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from numpy.lib.stride_tricks import sliding_window_view
from torch.utils.data import DataLoader, TensorDataset

SEED = 17
torch.manual_seed(SEED)
np.random.seed(SEED)

LEGAL_CORPUS = [
    "the non-disclosure agreement protects confidential information",
    "the contract protects confidential records",
    "the agreement lists payment obligations",
    "a non-disclosure clause limits disclosure",
    "the contract and agreement bind the company",
]


def tokenize(text):
    return re.findall(r"[a-z-]+", text.lower())


documents = [tokenize(text) for text in LEGAL_CORPUS]
print(f"Documents: {len(documents)}")
print("First tokenized document:", documents[0])

## 2 - A vocabulary turns repeated strings into stable addresses

A document is a list of words. The model needs one integer address for each retained word, so first flatten the list of documents into one stream for counting:

```text
for each document
    for each word in that document
        append the word
```

The leading expression in the equivalent comprehension comes first because it says what to collect:

```python
[word for document in documents for word in document]
```

`Counter` then records how often each word occurs. Frequency ordering makes the table deterministic and puts common teaching examples near the front; frequency is not a semantic coordinate.

Two views of the same table serve opposite directions:

- `itos[index]` maps an integer **index to string** for inspection.
- `stoi[word]` maps a **string to index** for model input.

Address zero is reserved for `<unk>`. Dictionary `.get(word, 0)` returns that fallback rather than raising an exception when a word is absent. The unknown row is a practical boundary, not a magic understanding of every unseen word.

**Predict:** Will `contract` and `agreement` receive nearby IDs because they are related? No. IDs are addresses; only training can organize their vectors.

In [ ]:
items = []
for document in documents:
    for word in document:
        items.append(word)

counts = Counter(items)
itos = ["<unk>"] + [word for word, _ in counts.most_common()]
stoi = {word: index for index, word in enumerate(itos)}
vocabulary_size = len(itos)

id_documents = [
    [stoi.get(word, stoi["<unk>"]) for word in document]
    for document in documents
]

print(f"Tokens counted: {len(items)}")
print(f"Vocabulary size: {vocabulary_size}")
print("First 10 index-to-string entries:", itos[:10])
print("First encoded document:", id_documents[0])
assert all(itos[index] == word for word, index in stoi.items())

## 3 - A sliding window manufactures prediction lessons

Choose two context words on each side of a target. The complete window is therefore:

```text
[left-2, left-1, TARGET, right+1, right+2]
```

Its width is `2 * context radius + 1`: four context positions plus one center target.

`sliding_window_view` exposes every consecutive window of that width without writing four nested indexing cases. For IDs `[1, 2, 3, 4, 5, 6]` and width five, it yields:

```text
[1, 2, 3, 4, 5]
[2, 3, 4, 5, 6]
```

Each document is windowed separately. That boundary matters: concatenating documents first would create fake lessons where the last words of one document predict the first words of the next.

Documents shorter than the full window cannot provide two neighbours on both sides, so they are skipped explicitly. Finally, any window containing `<unk>` is removed. That keeps this experiment focused, but a production pipeline would measure rather than silently discard unknown-heavy examples.

In [ ]:
CONTEXT_RADIUS = 2
window_width = 2 * CONTEXT_RADIUS + 1

per_document_windows = [
    sliding_window_view(np.asarray(ids), window_width)
    for ids in id_documents
    if len(ids) >= window_width
]
windows = np.concatenate(per_document_windows)
windows = windows[~(windows == stoi["<unk>"]).any(axis=1)]

print(f"Window width: {window_width}")
print(f"All windows shape: {windows.shape}")
for row in windows[:4]:
    print([itos[index] for index in row])

### Separate what the model sees from what it must predict

The center column is the answer. The remaining columns are the evidence.

`np.delete(..., axis=1)` removes the center **column from every row**; without `axis=1`, deletion would flatten or target the wrong dimension. Selecting `windows[:, CONTEXT_RADIUS]` keeps the center item from every row as the target.

PyTorch embedding layers require integer indices, so both arrays become `torch.long` tensors:

```text
complete windows: [number of lessons, 5]
context IDs:      [number of lessons, 4]
target IDs:       [number of lessons]
```

**Predict:** For a batch of eight lessons and embedding width twelve, what shape follows lookup? `[8, 4, 12]`.

In [ ]:
contexts = torch.tensor(
    np.delete(windows, CONTEXT_RADIUS, axis=1),
    dtype=torch.long,
)
targets = torch.tensor(windows[:, CONTEXT_RADIUS], dtype=torch.long)

print("Complete windows:", windows.shape)
print("Context tensor:", tuple(contexts.shape))
print("Target tensor:", tuple(targets.shape))
print(
    "First lesson:",
    [itos[index] for index in contexts[0].tolist()],
    "->",
    itos[targets[0]],
)

assert contexts.shape == (len(windows), 2 * CONTEXT_RADIUS)
assert targets.shape == (len(windows),)

## 4 - `DataLoader` changes delivery, not meaning

`TensorDataset` binds each input row to the target at the same row. Asking for item `i` always returns that pair together.

`DataLoader` groups pairs into batches and can shuffle the **lesson order** between epochs. It does not decide what an input or target means; CBOW, Skip-gram, and sequence-to-sequence will manufacture different tensors while keeping this delivery contract:

```text
for input_ids, target_ids in dataloader:
    ...
```

The small factory below is the only loader code used in the chapter. The current corpus fits in one batch, but the same function accepts a smaller batch size for a larger dataset. A fresh seeded generator keeps each teaching run deterministic.

In [ ]:
def make_dataloader(input_ids, target_ids, batch_size=None):
    dataset = TensorDataset(input_ids, target_ids)
    return DataLoader(
        dataset,
        batch_size=batch_size or len(dataset),
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
    )


dataloader = make_dataloader(contexts, targets)
batch_contexts, batch_targets = next(iter(dataloader))

print("Batch contexts:", tuple(batch_contexts.shape))
print("Batch targets:", tuple(batch_targets.shape))
print("First batch target:", itos[batch_targets[0]])
assert len(batch_contexts) == len(batch_targets)

## 5 - Crude attempt: give every context word an equal vote

An embedding table is a trainable matrix shaped:

```text
[vocabulary size, embedding width]
```

Passing IDs does not multiply by their numeric values. It retrieves the corresponding rows. A batch shaped `[B, C]` therefore becomes `[B, C, D]`, where `C` is the number of context positions and `D` is the embedding width.

CBOW means **continuous bag of words**. “Bag” is the warning label: averaging across the context axis gives one `[B, D]` vector, but the average remembers which rows participated, not where they appeared.

**Predict:** If you reverse one context, will its mean vector change?

In [ ]:
EMBEDDING_WIDTH = 12
probe_embedding = nn.Embedding(vocabulary_size, EMBEDDING_WIDTH)

one_context = contexts[:1]
reversed_context = one_context.flip(dims=[1])
one_vectors = probe_embedding(one_context)
original_mean = one_vectors.mean(dim=1)
reversed_mean = probe_embedding(reversed_context).mean(dim=1)

print("IDs:", one_context.tolist())
print("After lookup:", tuple(one_vectors.shape))
print("After context mean:", tuple(original_mean.shape))
print("Maximum difference after reversal:", (original_mean - reversed_mean).abs().max().item())

assert torch.allclose(original_mean, reversed_mean)

### Where `EmbeddingBag` fits

`nn.EmbeddingBag(..., mode="mean")` combines lookup and mean pooling. For a fixed-width 2D batch, it should match `nn.Embedding(...).mean(dim=1)` when both modules share the same weight table.

The compact layer is useful in real code, especially for variable-size bags. You will keep the two operations separate in the teaching model because the intermediate `[B, C, D]` tensor makes the information loss visible.

In [ ]:
embedding_bag = nn.EmbeddingBag(
    vocabulary_size,
    EMBEDDING_WIDTH,
    mode="mean",
)
with torch.no_grad():
    embedding_bag.weight.copy_(probe_embedding.weight)

manual_means = probe_embedding(contexts[:3]).mean(dim=1)
bag_means = embedding_bag(contexts[:3])

print("Manual mean shape:", tuple(manual_means.shape))
print("EmbeddingBag shape:", tuple(bag_means.shape))
print("Maximum difference:", (manual_means - bag_means).abs().max().item())
assert torch.allclose(manual_means, bag_means)

### Objective checkpoint 1 - CBOW

Before wrapping the path in a class, state the complete contract:

| Question | CBOW answer |
|---|---|
| Input IDs | `[B, C]`: several context positions |
| Target IDs | `[B]`: one center word per lesson |
| Vocabulary logits | `[B, V]`: one competition per lesson |
| Cross-entropy alignment | logits row `b` versus center ID `b` |
| Information combined | Which context rows appeared |
| Information discarded | Context order, because mean pooling is commutative |

**Predict:** Does the target have a time axis? Which exact operation removes the context-position
axis? If these answers are unclear, trace `[B, C] -> [B, C, D] -> [B, D]` before reading the model
class.

## 6 - Give replaceable models one small training contract

PyTorch already gives every model two familiar responsibilities:

- `__init__` creates and registers trainable layers.
- `forward` defines how one batch becomes logits.

The reusable loop still needs to ask one model-specific question: **which target positions should those logits be compared with?**

`TrainableLanguageModel` answers only that plumbing question. It does not choose layers, create data, own an optimizer, or hide backpropagation. Its default assumes one target ID per input row. The later encoder-decoder overrides only `targets_for_loss` because it predicts several positions.

```text
input IDs
-> model-specific forward
-> logits [..., vocabulary]
-> align target IDs [...]
-> flatten positions
-> cross-entropy
```

CBOW then supplies the first replaceable implementation:

```text
context IDs [B, C]
-> embedding rows [B, C, D]
-> equal mean [B, D]
-> hidden features [B, D/2]
-> vocabulary logits [B, V]
```

The base class is intentionally small enough to delete. Any model with an equivalent `training_loss` method can replace it.

In [ ]:
class TrainableLanguageModel(nn.Module):
    """Adapt model-specific logits to one visible cross-entropy contract."""

    def targets_for_loss(self, target_ids):
        return target_ids

    def training_loss(self, input_ids, target_ids, loss_function):
        logits = self(input_ids, target_ids)
        expected_ids = self.targets_for_loss(target_ids)
        flat_logits = logits.reshape(-1, logits.shape[-1])
        flat_targets = expected_ids.reshape(-1)
        return loss_function(flat_logits, flat_targets)


class CBOW(TrainableLanguageModel):
    def __init__(self, vocabulary_size, embedding_width):
        super().__init__()
        self.embedding = nn.Embedding(vocabulary_size, embedding_width)
        self.hidden = nn.Linear(embedding_width, embedding_width // 2)
        self.activation = nn.ReLU()
        self.output = nn.Linear(embedding_width // 2, vocabulary_size)

    def forward(self, context_ids, target_ids=None):
        token_vectors = self.embedding(context_ids)
        pooled_context = token_vectors.mean(dim=1)
        hidden_features = self.activation(self.hidden(pooled_context))
        return self.output(hidden_features)

### Trace one untrained batch before optimizing anything

Calling `model(batch)` invokes `forward` through PyTorch's module machinery. Prefer that call over `model.forward(batch)` because hooks and framework behavior live around the forward method.

The model returns `[batch, vocabulary]`. Each row is one target-word competition. Cross-entropy will compare those scores with one integer target per lesson.

In [ ]:
torch.manual_seed(SEED)
model = CBOW(vocabulary_size, EMBEDDING_WIDTH)
untrained_logits = model(batch_contexts)

print("Input contexts:", tuple(batch_contexts.shape))
print("Vocabulary logits:", tuple(untrained_logits.shape))
assert untrained_logits.shape == (len(batch_contexts), vocabulary_size)

## 7 - One `fit` loop should survive the model swap

Cross-entropy asks whether the expected token ID received a larger logit than every alternative. It applies the required log-softmax internally, so adding a separate softmax before the loss would be redundant and less stable.

One training step keeps the same strict order:

1. `zero_grad()` removes gradients accumulated by earlier steps.
2. `training_loss(...)` runs the model and aligns its targets.
3. `backward()` follows the computation graph and fills parameter gradients.
4. `step()` uses those gradients to update registered parameters.

The loop knows nothing about mean pooling, center words, recurrent state, or teacher forcing. Those mechanisms belong to the replaceable model. The loop does retain ordinary engineering responsibilities: training mode, sample-weighted loss accounting, and optimizer state across epochs.

In [ ]:
def fit_language_model(model, dataloader, loss_function, optimizer, epochs):
    history = []

    for _ in range(epochs):
        model.train()
        total_loss = 0.0

        for input_batch, target_batch in dataloader:
            optimizer.zero_grad()
            loss = model.training_loss(input_batch, target_batch, loss_function)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(input_batch)

        history.append(total_loss / len(dataloader.dataset))

    return history


criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.03)
losses = fit_language_model(
    model,
    dataloader,
    criterion,
    optimizer,
    epochs=180,
)

print(f"Loss: {losses[0]:.4f} -> {losses[-1]:.4f}")

### A falling loss is necessary evidence, not the whole story

With random, equally likely guesses across `V` words, cross-entropy is approximately `log(V)`. That baseline gives the loss scale meaning.

The plot checks optimization on this training corpus. It does **not** establish held-out generalization, broad language understanding, or production quality. This tiny run is a mechanism demonstration.

In [ ]:
random_guess_loss = math.log(vocabulary_size)

fig, axis = plt.subplots(figsize=(8, 4))
axis.plot(losses, color="#2a6f97", linewidth=2)
axis.axhline(
    random_guess_loss,
    color="#d97706",
    linestyle="--",
    label=f"random guess log(V) = {random_guess_loss:.2f}",
)
axis.set(title="CBOW training loss", xlabel="epoch", ylabel="cross-entropy")
axis.legend(frameon=False)
axis.grid(alpha=0.2)
plt.tight_layout()
plt.show()

print(f"Random-guess baseline: {random_guess_loss:.4f}")
print(f"Final training loss: {losses[-1]:.4f}")
assert losses[-1] < losses[0]
assert losses[-1] < random_guess_loss

## 8 - Follow one gradient back to the participating rows

An embedding lookup connects the loss only to rows retrieved by that batch. Repeated IDs contribute repeatedly to the same row; unused rows have no path from this batch's loss and receive zero embedding gradient.

The output layer is different: every vocabulary logit participates in cross-entropy, so this sparse-row statement applies to the **input embedding table**, not to every parameter in the model.

The next cell clears old gradients, performs one backward pass without an optimizer step, and compares used with unused embedding rows.

In [ ]:
model.zero_grad()
gradient_contexts = contexts[:2]
gradient_targets = targets[:2]
gradient_loss = model.training_loss(
    gradient_contexts,
    gradient_targets,
    criterion,
)
gradient_loss.backward()

gradient_norms = model.embedding.weight.grad.norm(dim=1)
used_ids = set(gradient_contexts.reshape(-1).tolist())
rows_with_gradient = {
    index for index, norm in enumerate(gradient_norms.tolist()) if norm > 0
}

print("Used rows:", [itos[index] for index in sorted(used_ids)])
print("Rows with gradients:", [itos[index] for index in sorted(rows_with_gradient)])
assert rows_with_gradient == used_ids

## 9 - The learned embedding weights are the vectors

`model.embedding.weight` is the trainable `[V, D]` table. Inspection should not extend the training graph, so:

- `detach()` returns a tensor that no longer tracks gradient history.
- `cpu()` would move GPU data to host memory; this notebook already runs on CPU, but the explicit move keeps the inspection pattern portable.
- dividing each row by its own norm creates unit vectors.

After normalization, a matrix-vector product is cosine similarity:

```text
all normalized rows @ one normalized query row
```

The query word itself would score one, so it is excluded. `<unk>` is also excluded because it received no meaningful unknown-word training in this corpus.

Nearest neighbours reveal geometry induced by this exact objective and corpus. They do not prove dictionary-like meaning.

In [ ]:
embedding_weights = model.embedding.weight.detach().cpu()
embedding_norms = embedding_weights.norm(dim=1, keepdim=True).clamp_min(1e-12)
normalized_embeddings = embedding_weights / embedding_norms


def most_similar(word, top_k=4):
    if word not in stoi:
        return f"'{word}' is not in the vocabulary"

    similarities = normalized_embeddings @ normalized_embeddings[stoi[word]]
    similarities = similarities.clone()
    similarities[stoi[word]] = -1
    similarities[stoi["<unk>"]] = -1
    neighbour_ids = similarities.topk(min(top_k, vocabulary_size - 2)).indices
    return [
        (itos[index], round(similarities[index].item(), 3))
        for index in neighbour_ids.tolist()
    ]


for word in ["contract", "agreement", "protects", "confidential"]:
    print(f"{word:>12} -> {most_similar(word)}")

### A 2D projection is a map, not the territory

The embedding rows live in twelve dimensions. To draw them, center the selected rows and use singular value decomposition to find two directions that preserve as much variance as possible.

This PCA-style view is deterministic and useful for orientation. It still discards dimensions, so apparent 2D distance is weaker evidence than cosine similarity measured in the original space. Labels can also make sparse regions look more meaningful than they are.

The most frequent words are shown because a crowded plot of every row would trade explanation for confetti.

In [ ]:
visible_count = min(16, vocabulary_size - 1)
visible_ids = torch.tensor(
    [stoi[word] for word, _ in counts.most_common(visible_count)]
)
visible_vectors = embedding_weights[visible_ids]
centered = visible_vectors - visible_vectors.mean(dim=0, keepdim=True)
_, _, right_singular_vectors = torch.linalg.svd(centered, full_matrices=False)
coordinates = centered @ right_singular_vectors[:2].T

fig, axis = plt.subplots(figsize=(10, 7))
axis.scatter(coordinates[:, 0], coordinates[:, 1], color="#2a6f97", s=55)
for point, token_id in zip(coordinates, visible_ids.tolist()):
    axis.annotate(itos[token_id], point, xytext=(5, 5), textcoords="offset points")
axis.axhline(0, color="#9ca3af", linewidth=0.7)
axis.axvline(0, color="#9ca3af", linewidth=0.7)
axis.set(
    title="Frequent embedding rows projected from 12D to 2D",
    xlabel="projection axis 1",
    ylabel="projection axis 2",
)
axis.grid(alpha=0.15)
plt.tight_layout()
plt.show()

## 10 - Flip the arrow: Skip-gram predicts neighbours from the center

CBOW asks:

```text
[left, left, right, right] -> center
```

A careful engineer can ask the reverse question: if the center row is useful, can it predict each neighbouring row?

```text
center -> left
center -> left
center -> right
center -> right
```

That reverse objective is Skip-gram. The original windows and vocabulary stay intact. Each CBOW lesson simply expands into four center-to-neighbour pairs. The `DataLoader` still emits `(input IDs, target IDs)`; only the tensor shapes and their meaning change.

**Predict:** With `N` windows and four context positions, how many Skip-gram lessons appear? `4N`.

In [ ]:
skipgram_inputs = targets.repeat_interleave(contexts.shape[1])
skipgram_targets = contexts.reshape(-1)
skipgram_dataloader = make_dataloader(skipgram_inputs, skipgram_targets)

print("CBOW lessons:", len(contexts))
print("Skip-gram lessons:", len(skipgram_inputs))
print(
    "First Skip-gram lesson:",
    itos[skipgram_inputs[0]],
    "->",
    itos[skipgram_targets[0]],
)

assert len(skipgram_inputs) == len(contexts) * contexts.shape[1]
assert len(skipgram_inputs) == len(skipgram_targets)

### Objective checkpoint 2 - Skip-gram

The loader still returns two tensors, but their rank and semantic roles changed:

| Question | Skip-gram answer |
|---|---|
| Input IDs | `[B]`: one repeated center ID per pair |
| Target IDs | `[B]`: one neighbouring word per pair |
| Vocabulary logits | `[B, V]`: one competition per pair |
| Cross-entropy alignment | logits row `b` versus neighbour ID `b` |
| Information available | Center-word identity |
| Information unavailable | Which neighbour position or direction produced the target |

**Predict:** Which CBOW axis disappeared? Why can the same center ID appear with several different
targets? The shared fit loop survives because it consumes aligned logits and IDs, not because the
two objectives mean the same thing.

### The model swap is smaller than the objective swap

CBOW looked up several rows and averaged them. Skip-gram receives one center ID, so there is no context axis to pool. The hidden projection and vocabulary competition can keep the same roles:

```text
center IDs [B]
-> embedding rows [B, D]
-> hidden features [B, D/2]
-> vocabulary logits [B, V]
```

The shared base class, loader factory, loss function, and fit loop do not change. This teaching version uses full-vocabulary cross-entropy so you can see the exact contract. Classical large-corpus Word2Vec usually uses negative sampling or hierarchical softmax to avoid scoring every vocabulary item.

In [ ]:
class SkipGram(TrainableLanguageModel):
    def __init__(self, vocabulary_size, embedding_width):
        super().__init__()
        self.embedding = nn.Embedding(vocabulary_size, embedding_width)
        self.hidden = nn.Linear(embedding_width, embedding_width // 2)
        self.activation = nn.ReLU()
        self.output = nn.Linear(embedding_width // 2, vocabulary_size)

    def forward(self, center_ids, target_ids=None):
        center_vectors = self.embedding(center_ids)
        hidden_features = self.activation(self.hidden(center_vectors))
        return self.output(hidden_features)


torch.manual_seed(SEED)
skipgram_model = SkipGram(vocabulary_size, EMBEDDING_WIDTH)
skipgram_optimizer = torch.optim.Adam(skipgram_model.parameters(), lr=0.03)
skipgram_losses = fit_language_model(
    skipgram_model,
    skipgram_dataloader,
    criterion,
    skipgram_optimizer,
    epochs=220,
)

with torch.inference_mode():
    probe_center = targets[:1]
    neighbour_logits = skipgram_model(probe_center)
    predicted_ids = neighbour_logits.topk(contexts.shape[1], dim=1).indices[0]

print(f"Skip-gram loss: {skipgram_losses[0]:.4f} -> {skipgram_losses[-1]:.4f}")
print("Center:", itos[probe_center.item()])
print("Top predicted neighbours:", [itos[index] for index in predicted_ids.tolist()])

assert skipgram_losses[-1] < skipgram_losses[0]
assert skipgram_losses[-1] < math.log(vocabulary_size)
assert neighbour_logits.shape == (1, vocabulary_size)

## 11 - Both static objectives still miss token roles

CBOW improved the center-word objective and organized the embedding table. Skip-gram reversed the arrow, but it still starts from one static center row and predicts neighbouring words independently. Neither objective creates a separate contextual state for each token position.

CBOW exposes the sharper architectural failure because it averages context rows. Addition is commutative, so these contexts are identical to it:

```text
the contract protects confidential
confidential protects contract the
```

The model has no position input and creates only one pooled vector for the whole bag. Reordering a context must therefore leave both its pooled representation and vocabulary logits unchanged.

This is stronger than “the model is small.” More data, more epochs, or wider embeddings cannot repair information discarded before prediction.

In [ ]:
model.eval()
ordered_context = contexts[:1]
permuted_context = ordered_context[:, torch.tensor([3, 1, 0, 2])]

with torch.no_grad():
    ordered_vectors = model.embedding(ordered_context).mean(dim=1)
    permuted_vectors = model.embedding(permuted_context).mean(dim=1)
    ordered_logits = model(ordered_context)
    permuted_logits = model(permuted_context)

representation_difference = (ordered_vectors - permuted_vectors).abs().max().item()
logit_difference = (ordered_logits - permuted_logits).abs().max().item()

print("Ordered:", [itos[index] for index in ordered_context[0].tolist()])
print("Permuted:", [itos[index] for index in permuted_context[0].tolist()])
print(f"Maximum representation difference: {representation_difference:.8f}")
print(f"Maximum logit difference: {logit_difference:.8f}")

assert torch.allclose(ordered_vectors, permuted_vectors)
assert torch.allclose(ordered_logits, permuted_logits)

## 12 - One target token is not a target sequence

Skip-gram changed the direction of one-token prediction. It still emits one independent vocabulary competition per lesson. Translation, summarization, and continuation need an ordered target with several positions.

Reuse each five-token window without downloading another dataset:

```text
source: [token 0, token 1, token 2]
target: [<bos>, token 3, token 4, <eos>]
```

This is a continuation task, not translation. It is enough to expose the encoder-decoder contract while keeping the legal corpus, vocabulary rows, loader factory, and loss function familiar.

`<bos>` tells the decoder where generation starts. `<eos>` gives it a learnable stopping target. Every sequence has the same length here, so padding would add machinery without teaching a new fact.

In [ ]:
BOS_ID = vocabulary_size
EOS_ID = vocabulary_size + 1
sequence_itos = itos + ["<bos>", "<eos>"]
sequence_vocabulary_size = len(sequence_itos)

window_ids = torch.tensor(windows, dtype=torch.long)
source_sequences = window_ids[:, :3]
continuations = window_ids[:, 3:]
sequence_targets = torch.cat(
    [
        torch.full((len(windows), 1), BOS_ID, dtype=torch.long),
        continuations,
        torch.full((len(windows), 1), EOS_ID, dtype=torch.long),
    ],
    dim=1,
)
sequence_dataloader = make_dataloader(source_sequences, sequence_targets)

print("Source shape:", tuple(source_sequences.shape))
print("Target shape:", tuple(sequence_targets.shape))
print("Source:", [sequence_itos[index] for index in source_sequences[0].tolist()])
print("Target:", [sequence_itos[index] for index in sequence_targets[0].tolist()])

assert source_sequences.shape == (len(windows), 3)
assert sequence_targets.shape == (len(windows), 4)

### Objective checkpoint 3 - encoder-decoder sequence prediction

The target now has positions. Write the shift before looking at the recurrent implementation:

| Question | Encoder-decoder answer |
|---|---|
| Source IDs | `[B, S]`: an ordered source sequence |
| Full target IDs | `[B, T]`: `<bos>`, continuation, `<eos>` |
| Decoder inputs | `[B, T-1]`: target without the final token |
| Vocabulary logits | `[B, T-1, V]`: one competition per predicted position |
| Loss targets | `[B, T-1]`: target without `<bos>` |
| Information available | Ordered source compressed into one recurrent hidden state, plus the gold target prefix |
| Information unavailable | Direct retrieval of individual source positions; this model has no attention |

**Predict:** Which axes flatten for cross-entropy, and which axis must remain the vocabulary axis?
This is a compact preview of encoder-decoder alignment. Stage 04 will rebuild the mechanism with
cross-attention rather than treating this one-state bottleneck as the final architecture.

### The model owns sequence alignment; the loop stays boring

The encoder reads all source embeddings in order and returns a hidden state. The decoder starts from that state and reads the gold target prefix:

```text
encoder: source IDs [B, S] -> hidden [1, B, H]
decoder input: target IDs without final token [B, T-1]
decoder logits: [B, T-1, V]
loss targets: target IDs without <bos> [B, T-1]
```

That shifted alignment means each decoder position predicts the **next** target token. Feeding the gold previous token during training is teacher forcing. It makes the first mechanism easy to inspect, but creates a train-versus-generation mismatch that later chapters must measure.

Only `targets_for_loss` changes the shared training contract. The fit loop still sees two tensors, asks the model for a loss, calls `backward`, and lets the optimizer step.

In [ ]:
class EncoderDecoderLM(TrainableLanguageModel):
    def __init__(self, vocabulary_size, embedding_width, hidden_width):
        super().__init__()
        self.embedding = nn.Embedding(vocabulary_size, embedding_width)
        self.encoder = nn.GRU(embedding_width, hidden_width, batch_first=True)
        self.decoder = nn.GRU(embedding_width, hidden_width, batch_first=True)
        self.output = nn.Linear(hidden_width, vocabulary_size)

    def encode(self, source_ids):
        _, hidden = self.encoder(self.embedding(source_ids))
        return hidden

    def decode(self, decoder_input_ids, hidden):
        decoder_states, hidden = self.decoder(
            self.embedding(decoder_input_ids),
            hidden,
        )
        return self.output(decoder_states), hidden

    def forward(self, source_ids, target_ids=None):
        if target_ids is None:
            raise ValueError("Training requires target IDs for teacher forcing.")
        hidden = self.encode(source_ids)
        logits, _ = self.decode(target_ids[:, :-1], hidden)
        return logits

    def targets_for_loss(self, target_ids):
        return target_ids[:, 1:]


torch.manual_seed(SEED)
sequence_model = EncoderDecoderLM(
    sequence_vocabulary_size,
    embedding_width=EMBEDDING_WIDTH,
    hidden_width=24,
)
sequence_optimizer = torch.optim.Adam(sequence_model.parameters(), lr=0.03)
sequence_losses = fit_language_model(
    sequence_model,
    sequence_dataloader,
    criterion,
    sequence_optimizer,
    epochs=400,
)

with torch.inference_mode():
    teacher_forced_logits = sequence_model(source_sequences, sequence_targets)
    teacher_forced_predictions = teacher_forced_logits.argmax(dim=-1)
    expected_sequence_ids = sequence_targets[:, 1:]
    teacher_forced_accuracy = (
        teacher_forced_predictions == expected_sequence_ids
    ).float().mean().item()

print(f"Seq2Seq loss: {sequence_losses[0]:.4f} -> {sequence_losses[-1]:.4f}")
print("Teacher-forced token accuracy:", f"{teacher_forced_accuracy:.0%}")
print("Sequence logits:", tuple(teacher_forced_logits.shape))

assert sequence_losses[-1] < sequence_losses[0]
assert sequence_losses[-1] < math.log(sequence_vocabulary_size)
assert teacher_forced_accuracy >= 0.9

### Teacher forcing worked - can the decoder use its own answer?

Training supplied the correct previous target token. Generation cannot. A greedy decoder must feed its own highest-logit token back into the next step:

```text
<bos> -> predicted token -> predicted token -> ... -> <eos>
```

This is the smallest honest generation check. It is not beam search, and exact reproduction would show memorization of eight tiny windows rather than general translation ability.

The second check reverses one source. Unlike CBOW's mean, a GRU updates state step by step, so reversing tokens should change the encoder state. Order sensitivity is necessary for sequence modeling; it is not sufficient evidence of a good model.

In [ ]:
def greedy_decode(model, source_ids, bos_id, eos_id, max_steps):
    model.eval()
    hidden = model.encode(source_ids)
    current_ids = torch.full(
        (len(source_ids), 1),
        bos_id,
        dtype=torch.long,
    )
    generated = []

    with torch.inference_mode():
        for _ in range(max_steps):
            logits, hidden = model.decode(current_ids, hidden)
            next_ids = logits[:, -1].argmax(dim=-1)
            generated.append(next_ids)
            current_ids = next_ids.unsqueeze(1)
            if bool((next_ids == eos_id).all()):
                break

    return torch.stack(generated, dim=1)


generated_ids = greedy_decode(
    sequence_model,
    source_sequences[:1],
    BOS_ID,
    EOS_ID,
    max_steps=sequence_targets.shape[1] - 1,
)

with torch.inference_mode():
    ordered_hidden = sequence_model.encode(source_sequences[:1])
    reversed_hidden = sequence_model.encode(source_sequences[:1].flip(dims=[1]))

expected_words = [
    sequence_itos[index]
    for index in sequence_targets[0, 1:].tolist()
]
generated_words = [
    sequence_itos[index]
    for index in generated_ids[0].tolist()
]
hidden_difference = (ordered_hidden - reversed_hidden).abs().max().item()

print("Expected continuation:", expected_words)
print("Greedy continuation:", generated_words)
print(f"Hidden-state difference after reversal: {hidden_difference:.6f}")

assert generated_ids.shape[0] == 1
assert generated_ids.shape[1] >= 1
assert hidden_difference > 0

### Static rows, pooled context, and recurrent state are different objects

| Object | What it knows | What it cannot know here |
|---|---|---|
| Static embedding row | Which vocabulary address was retrieved | The sentence, position, or current meaning |
| CBOW pooled vector | Which rows appeared in the local bag | Their order or a separate state for each position |
| Skip-gram center vector | Which center row must predict neighbours | Which neighbour position is being predicted |
| Encoder hidden state | The source tokens in recurrent order | Direct access to every earlier state once compressed |
| Transformer contextual state | Token identity plus information gathered for one position | Nothing automatically; access depends on masks, position, attention, and training |

CBOW and Skip-gram are Word2Vec objectives: they learn useful static rows, but they are not free-running language models. The encoder-decoder is a conditional sequence model, yet this tiny version sees gold previous target tokens during training and has no attention.

## Toy-to-real bridge

| Teaching mechanism | Production counterpart | What changes | What does not change |
|---|---|---|---|
| `make_dataloader(input_ids, target_ids)` | Task-specific collation and distributed loading | Padding, masks, sharding, and scale | Batches still preserve input-target alignment |
| `training_loss` contract | Framework training step | Mixed precision, accumulation, metrics, and distributed reduction | Logits must align with target positions |
| CBOW full-vocabulary loss | Word2Vec CBOW with negative sampling | Efficient sampled objective and corpus scale | Context predicts center |
| Skip-gram full-vocabulary loss | Word2Vec Skip-gram with negative sampling | Efficient sampled objective and subsampling | Center predicts neighbours |
| GRU encoder-decoder | Translation or summarization model | Variable lengths, masks, attention, and beam search | Source conditions target generation |
| Recurrent bottleneck | Cross-attention over source states | Decoder retrieves rather than compressing everything once | Target tokens remain autoregressive |

### Your turn

1. Change `CONTEXT_RADIUS` from two to one. Predict how many Skip-gram pairs each window creates.
2. Replace CBOW mean pooling with sum pooling. Reordering will still be invisible; explain why.
3. Set the sequence source to two tokens instead of three and retrain. Compare the continuation accuracy without claiming the tiny corpus generalizes.

These drills change one known variable. None turns the toy corpus into a production language model.

## 13 - Evidence scorecard and handoff

| Opening question | Evidence produced here |
|---|---|
| How does text become lessons? | Per-document windows produced CBOW, Skip-gram, and source-target sequence tensors |
| What stayed reusable? | One `DataLoader` factory, `training_loss` contract, cross-entropy, and `fit_language_model` loop trained all three models |
| What changed for Skip-gram? | The center and context swapped roles; model input became one ID and targets became neighbouring IDs |
| What changed for sequence-to-sequence? | Targets gained a time axis, the encoder produced recurrent state, and the decoder predicted shifted target positions |
| Are embeddings learned weights? | Gradient inspection identified exactly the retrieved CBOW input rows |
| Did optimization beat useful baselines? | All three final losses were asserted below their initial values and random-guess baselines |
| Did the models acquire production semantics or generation quality? | Not established; every result came from a tiny training corpus |
| What breaks next? | Static objectives lose token roles; the recurrent encoder compresses the source into one bottleneck |

### Coverage ledger

**Built and measured:** token counting, per-document windows, one reusable loader factory, a documented model training contract, explicit CBOW, generic optimization, gradient participation, cosine neighbours, PCA projection, Skip-gram pair construction, full-softmax Skip-gram, fixed-length encoder-decoder training, greedy decoding, sequence accuracy, recurrent order sensitivity, and CBOW permutation invariance.

**Explained or illustrated:** Word2Vec objective direction, negative-sampling boundary, teacher forcing, static versus contextual representations, recurrent source bottleneck, and toy-to-production mappings.

**Named and out of scope:** large-corpus embedding quality, subsampling frequent words, negative sampling implementation, variable-length padding and masks, scheduled sampling, attention, beam search, held-out model selection, and production translation. They belong later because implementing all of them here would hide the reusable skeleton.

**Next:** [Stage 04: Transformers](../../04-transformers/README.md) gives every position its own state. Its encoder-decoder chapter later replaces the one-vector source bottleneck with cross-attention.